In [1]:
import getpass
import os
def _set_if_undefined(var: str) -> None:
    if os.environ.get(var):
        return
    os.environ[var] = getpass.getpass(var)


_set_if_undefined("OPENAI_API_KEY")
#sk-afda581a0d314c64b897b4df1b8bdc05,这里为了复制方便，实际开发不使用明码
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

In [2]:

import sentence_transformers  # ★ 必须先导入，避免 Windows 上 DLL 加载顺序导致 segfault
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.openai_like import OpenAILike
import chromadb
from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    StorageContext,
    Settings,
)
from llama_index.core.node_parser import SentenceSplitter
from llama_index.vector_stores.chroma import ChromaVectorStore


In [5]:

# ---- 1. 全局配置：embedding / LLM / 切分器 ----
# Settings 是 LlamaIndex 的全局默认，配一次后面所有步骤都用它
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-base-zh-v1.5")
Settings.llm = OpenAILike(
    model="deepseek-v4-pro",
    api_base="https://api.deepseek.com",   # 注意带 /v1
    is_chat_model=True,        # ★ 必加,否则会去打 completions 口然后报错
    context_window=128000,
    max_tokens=1024*8,
)
Settings.node_parser = SentenceSplitter(chunk_size=512, chunk_overlap=0)

pytorch_model.bin:   0%|          | 0.00/409M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/409M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/110k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/439k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:

# ---- 2. 加载 PDF（每页通常是一个 Document）----
documents = SimpleDirectoryReader("./data").load_data()
print(repr(documents[0].text[:300]))   # ★ 先看这个

'广州大学研究生学位论文中期报告表\n研 究 生\n姓 名 王克 所在\n学院\n计算机科学与\n网络工程学院\n导\n师 朱恩强 学\n号 2112406169\n专 业 软件工程 研究方向 人工智能\n论文题目：通过有效性感知 Transformer 解决约束满足问题\n论文工作中期报告时间：2026.5.25 地点：A3-308\n参加论文中期报告教师（要求副高以上职称的教师不少于三人）签名：\n论文工作情况（包括已取得的阶段性结果和存在的问题，下一步的工作计划等）：\n本项目自启动以来进展顺利，已圆满完成核心算法设计、全量对比实验及底层机\n制解析，取得了一系列突破性的阶段成果。我们创新性地提出了一种融合关系结构偏\n'


In [7]:

print(f"loaded {len(documents)} document pages")

# ---- 3. 连接 Chroma（持久化到本地 ./chroma_db）----
chroma_client = chromadb.PersistentClient(path="./chroma_db")
collection = chroma_client.get_or_create_collection("my_rag")
vector_store = ChromaVectorStore(chroma_collection=collection)#把 Chroma 的 collection 包装成 LlamaIndex 认识的"转接头"
storage_context = StorageContext.from_defaults(vector_store=vector_store)#StorageContext—— LlamaIndex 用来回答"东西到底存哪儿"的对象,

# ---- 4. 建索引：切分 → 向量化 → 写入 Chroma，这一步一气呵成 ----
index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
)

# ---- 5. 检索 top-5 + Claude 生成 ----
query_engine = index.as_query_engine(similarity_top_k=5)
response = query_engine.query("这份中期报告表的研究生是谁")
print("\n=== 回答 ===")
print(response)

# 调试：看看到底检索到了哪 5 个片段
for i, node in enumerate(response.source_nodes):
    print(f"\n--- chunk {i}  (score={node.score:.3f}) ---")
    print(node.text[:200])

loaded 9 document pages


InvalidArgumentError: Collection expecting embedding with dimension of 512, got 768

In [ ]:
QA_PAIRS = [
    # ===== 直接查找型（12 个）：答案在文档里明确写着 =====
    {"question": "这份材料的研究生姓名是什么？", "reference": "王克"},
    {"question": "王克的校内指导教师是谁？", "reference": "朱恩强"},
    {"question": "王克的学号是多少？", "reference": "2112406169"},
    {"question": "学位论文的题目是什么？", "reference": "通过有效性感知 Transformer 解决约束满足问题"},
    {"question": "王克所在的学院是哪个？", "reference": "计算机科学与网络工程学院"},
    {"question": "王克的专业是什么？", "reference": "软件工程"},
    {"question": "中期考核答辩小组的组长是谁？", "reference": "强小利"},
    {"question": "该模型在 9x9 数独基准上达到的求解精度是多少？", "reference": "99.94%"},
    {"question": "王克把英文论文投稿到了哪个学术会议？", "reference": "NeurIPS 2026"},
    {"question": "投稿的英文论文有多少页？", "reference": "45 页"},
    {"question": "王克是哪一年级的研究生？", "reference": "2024 级"},
    {"question": "学位论文开题报告的通过时间是什么时候？", "reference": "2025年12月31日"},

    # ===== 需要综合型（5 个）：要跨段落/跨文档聚合或筛选 =====
    {"question": "中期考核答辩小组由哪几位成员组成？",
     "reference": "共4位：组长强小利、成员陈智华和许鹏、记录员孙思"},
    {"question": "该研究的模型成功迁移到了哪些组合优化问题上？",
     "reference": "16x16 复杂数独、10x10 二元谜题、Erdős-Rényi 随机图着色"},
    {"question": "王克的研究目前面临哪些主要挑战或局限？",
     "reference": "辅助约束损失与主目标产生梯度冲突；模型高度依赖人工预定义的离散关系矩阵，尚不能从非结构化场景自动提取关系图"},
    {"question": "王克达到毕业要求的后续计划是什么？",
     "reference": "将45页英文论文翻译成中文并少量扩充润色，预计研三第一学期完成学位论文"},
    {"question": "论文中期报告与中期考核的答辩日期和地点分别是？两者是否一致？",
     "reference": "均为2026年5月25日、地点A3-308，两者一致"},

    # ===== 诚实性探针（3 个）：文档里没有答案，正确表现是“说不知道”而非硬编 =====
    {"question": "王克的本科毕业院校是哪所大学？", "reference": "文档中未提及"},
    {"question": "这篇论文除王克外的合作作者有哪些？", "reference": "文档中未提及"},
    {"question": "王克的联系邮箱或电话是多少？", "reference": "文档中未提及"},
]

In [ ]:


import os
import nest_asyncio  # Jupyter 里跑需要它；纯脚本下加着也无害
nest_asyncio.apply()

TOP_K = 5
query_engine = index.as_query_engine(similarity_top_k=TOP_K)



# ============ C. 跑 RAG，收集每个问题的 答案 + 检索片段 ============
def run_rag(question: str):
    """返回 (答案文本, 检索到的片段列表)"""
    resp = query_engine.query(question)
    answer = str(resp)
    contexts = [n.node.get_content() for n in resp.source_nodes]
    return answer, contexts


records = []
for i, qa in enumerate(QA_PAIRS):
    ans, ctxs = run_rag(qa["question"])
    records.append({
        "user_input": qa["question"],      # RAGAS 字段名：问题
        "retrieved_contexts": ctxs,         # RAGAS 字段名：检索片段（list）
        "response": ans,                    # RAGAS 字段名：RAG 给出的答案
        "reference": qa["reference"],       # RAGAS 字段名：标准答案（旧版叫 ground_truth）
    })
    print(f"[{i+1}/{len(QA_PAIRS)}] {qa['question']}  ->  {ans[:40]}...")


In [ ]:

# ============ D. 构建评审模型（DeepSeek 当 judge + 本地 bge 当 embedding）============
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(
    model="deepseek-v4-pro",
    base_url="https://api.deepseek.com/",
    temperature=0,   # ★ judge 必须 temperature=0，否则分数不可复现
))
evaluator_emb = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="BAAI/bge-small-zh-v1.5")
)


# ============ E. 跑评估 ============
from ragas import EvaluationDataset, evaluate, RunConfig
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

evaluation_dataset = EvaluationDataset.from_list(records)

result = evaluate(
    dataset=evaluation_dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=evaluator_llm,
    embeddings=evaluator_emb,
    run_config=RunConfig(max_workers=4, timeout=120),  # DeepSeek 限流时把 max_workers 调小
)

print("\n========== 基准分数（均值）==========")
print(result)

# 存档：聚合分 + 每条样本明细，连同关键配置一起记录，才有对照价值
df = result.to_pandas()
df.to_csv("ragas_baseline_per_sample.csv", index=False)

with open("ragas_baseline_summary.txt", "a", encoding="utf-8") as f:
    f.write("=== RAGAS Baseline ===\n")
    f.write(f"样本数: {len(records)}\n")
    f.write("配置（必须记录，否则没法对照）:\n")
    f.write("  embedding = BAAI/bge-small-zh-v1.5\n")
    f.write("  generator = deepseek-v4-pro\n")
    f.write("  judge     = deepseek-v4-pro (temperature=0)\n")
    f.write(f"  chunk_size=512, chunk_overlap=50, top_k={TOP_K}\n\n")
    f.write(str(result) + "\n")

print("\n已保存：ragas_baseline_per_sample.csv（逐条）/ ragas_baseline_summary.txt（汇总+配置）")